## Dependencies and Imports

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os
os.chdir('/content/drive/Othercomputers/My Laptop/Work/Grievance Redressal Project')

In [6]:
# # Install dependencies
!pip install yake keybert pytextrank spacy
!python -m spacy download en_core_web_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.4/91.4 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.5/360.5 kB 23.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 117.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [7]:
import numpy as np
import pandas as pd
import matplotlib as plt
import yake
from keybert import KeyBERT
import spacy
import pytextrank
import torch
from tqdm import tqdm

/usr/local/lib/python3.12/dist-packages


## Dataset

In [9]:
df = pd.read_csv('data/grievance_dataset_domain_classification.csv')

In [10]:
df.shape

(37854, 9)

In [11]:
df.sample(5)

,_id,org_code,recvd_date,closing_date,sex,state,grievance_text,remarks_text,grievance_text_processed
35354,DDESW/E/2023/0000271,DDESW,2023-01-30 00:37:19.047000+00:00,2023-02-22 00:00:00+00:00,Male,MP,Ex Servicemen Welfare >> Service Related >> Ou...,1. Refer to CPGRAMS XDXSX/X/X0X3X0X0X2X1 dat...,ex servicemen welfare service related outstand...
25911,MOLBR/E/2023/0007411,MOLBR,2023-01-21 14:47:29.097000+00:00,2023-01-24 00:00:00+00:00,Male,HR,Labour and Employment >> Transfer related issu...,Member is advised to submit the rejection let...,labour employment transfer related issues tran...
19401,DOPAT/E/2023/0000964,DOPAT,2023-01-16 17:01:42.137000+00:00,2023-02-17 00:00:00+00:00,Male,MP,Personnel and Training >> Staff Selection Comm...,Madam/Sir\r\n\r\nYour suggestion has been not...,personnel training staff selection commission ...
22836,MOLBR/E/2023/0006489,MOLBR,2023-01-19 11:15:52.640000+00:00,2023-01-31 00:00:00+00:00,Male,MH,Labour and Employment >> Others (EPFO)\r\n\r\n...,It is informed that annual account for financ...,labour employment others epfo name address est...
37331,MOLBR/E/2023/0010691,MOLBR,2023-01-31 16:25:16.860000+00:00,2023-02-08 00:00:00+00:00,Male,MH,Labour and Employment >> Pension >> Settlement...,"Sir, As per this office record your service i...",labour employment pension settlement pensionde...


In [ ]:
df_top_20 = df[:20]

## YAKE and TextRank Extraction

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Configure SpaCy to use GPU
spacy.prefer_gpu()
nlp = spacy.load("en_core_web_sm")
nlp.add_pipe("textrank")

# Initialize models
kw_extractor = yake.KeywordExtractor(lan="en", n=2, top=10)

#  YAKE Implementation
# def extract_yake(text):
#     if pd.isna(text) or not text: return []
#     keywords = kw_extractor.extract_keywords(str(text))
#     return [kw[0] for kw in keywords]

def extract_yake(texts):
    results = []
    for text in tqdm(texts, desc="Processing YAKE"):
        if pd.isna(text) or text == "":
            results.append([])
            continue
        keywords = kw_extractor.extract_keywords(str(text))
        results.append([kw[0] for kw in keywords])
    return results

# TextRank Implementation
# def extract_textrank(text):
#     if pd.isna(text) or not text: return []
#     doc = nlp(str(text))
#     return [phrase.text for phrase in doc._.phrases[:5]]

def extract_textrank(texts):
    results = []
    # Using nlp.pipe for faster processing in SpaCy
    for doc in tqdm(nlp.pipe(texts, batch_size=256), total=len(texts), desc="Processing TextRank"):
        results.append([phrase.text for phrase in doc._.phrases[:5]])
    return results

# Apply to DataFrame
# df_top_20['yake_keywords'] = df_top_20['grievance_text_processed'].apply(extract_yake)
# df_top_20['textrank_keywords'] = df_top_20['grievance_text_processed'].apply(extract_textrank)
df['yake_keywords'] = extract_yake(df['grievance_text_processed'].tolist())
df['textrank_keywords'] = extract_textrank(df['grievance_text_processed'].astype(str).tolist())

Using device: cuda


Processing TextRank: 100%|██████████| 37854/37854 [06:55<00:00, 91.01it/s] 


In [13]:
df.shape

(37854, 11)

In [14]:
df.head(5)

,_id,org_code,recvd_date,closing_date,sex,state,grievance_text,remarks_text,grievance_text_processed,yake_keywords,textrank_keywords
0,MORLY/E/2023/0000001,MORLY,2023-01-01 00:00:19.977000+00:00,2023-01-04 00:00:00+00:00,Male,WB,"Railways, ( Railway Board) >> Miscellaneous\r\...","As per railway record, there is no authoriz...",railways railway board miscellaneous railway b...,"[railway board, board railway, bhaskar mitra, ...",[railways railway board miscellaneous railway ...
1,MOLBR/E/2023/0000001,MOLBR,2023-01-01 00:01:45.593000+00:00,2023-01-12 00:00:00+00:00,Male,TS,Labour and Employment >> PF Withdrawal >> Othe...,"Sir/Madam, With reference to Grievance no. XO...",labour employment pf withdrawal others name ad...,"[final settlement, skechers south, south asia,...","[pf final settlement, south asia pvt ltd sir, ..."
2,MOLBR/E/2023/0000002,MOLBR,2023-01-01 00:02:07.247000+00:00,2023-01-06 00:00:00+00:00,Male,MH,Labour and Employment >> Pension >> Others\r\n...,Please submit establishment clarification let...,labour employment pension others name address ...,"[kanwar narender, narender singh, karr sakta, ...","[eps xoxtxixuxixn hua hai bulki mera, hee fasa..."
3,MODEF/E/2023/0000001,MODEF,2023-01-01 00:04:02.500000+00:00,2023-01-03 00:00:00+00:00,Female,JK,Defence >> Canteen Stores Depot related >> Non...,Son is eligible for dependent CSD Smart Card ...,defence canteen stores depot related non entry...,"[grocery dependent, dependent card, card renew...","[grocery dependent card son son attains, groce..."
4,MEAPD/E/2023/0000001,MEAPD,2023-01-01 00:04:30.550000+00:00,2023-01-09 00:00:00+00:00,Male,TS,External Affairs >> Others\r\n----------------...,"As per the HCI, Wellington, the service sough ...",external affairs others danapuneni sreekanth r...,"[ministry external, external affairs, tution f...","[ministry external affairs indian government, ..."


In [15]:
df.to_csv('data/grievance_dataset_keyword_extraction.csv', index=False)

## LexRank Summarization

In [ ]:
# Install library
# !pip install sumy

import nltk
# nltk.download('punkt')
# nltk.download('punkt_tab')

from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lex_rank import LexRankSummarizer

def extractive_summarize(text, sentence_count=2):
    if not text or len(text.split()) < 20: return text
    parser = PlaintextParser.from_string(text, Tokenizer("english"))
    summarizer = LexRankSummarizer()
    summary = summarizer(parser.document, sentence_count)
    return " ".join([str(sentence) for sentence in summary])

# Apply to your dataframe
df['extractive_summary'] = df['grievance_text'].apply(extractive_summarize)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
/usr/local/lib/python3.12/dist-packages/sumy/summarizers/lex_rank.py:167: RuntimeWarning: invalid value encountered in divide
  next_p /= numpy.linalg.norm(next_p)
